In [2]:
# !pip install -q transformers datasets scikit-learn torch accelerate

import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, f1_score
import torch
from torch import nn
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)

# --------------- CONFIG -----------------
DATA_PATH = "/kaggle/input/weighted-v1/blp25_hatespeech_subtask_1A_train.tsv"  # your upload
DATA_PATH_2= "/kaggle/input/weighted-v1/blp25_hatespeech_subtask_1A_dev.tsv"
TEXT_COL = "text"   # <-- change if needed (e.g., "tweet", "sentence", etc.)
LABEL_COL = "label" # as you specified
MODEL_NAME = "csebuetnlp/banglabert"  # or "roberta-base"
MAX_LEN = 128
TEST_SIZE = 0.15
RANDOM_STATE = 42
BATCH_SIZE = 16
NUM_EPOCHS = 5
LR = 2e-5
WEIGHT_DECAY = 0.01
# ----------------------------------------

# Load TSV
train_df = pd.read_csv(DATA_PATH, sep="\t", keep_default_na=False)
val_df = pd.read_csv(DATA_PATH_2, sep="\t", keep_default_na=False)
# If you're unsure of the text column name, uncomment below to guess it:
# if TEXT_COL not in df.columns:
#     candidate_text_cols = [c for c in df.columns if c != LABEL_COL and df[c].dtype == object]
#     if not candidate_text_cols:
#         raise ValueError("Couldn't find a text column. Set TEXT_COL to the correct name.")
#     TEXT_COL = candidate_text_cols[0]
#     print("Guessed TEXT_COL =", TEXT_COL)

# Ensure labels are integers 0..(num_labels-1)
# If labels are strings, map them to ids
if train_df[LABEL_COL].dtype == object:
    label2id = {lbl: i for i, lbl in enumerate(sorted(train_df[LABEL_COL].unique()))}
    id2label = {i: lbl for lbl, i in label2id.items()}
    train_df[LABEL_COL] = train_df[LABEL_COL].map(label2id)
    val_df[LABEL_COL] = val_df[LABEL_COL].map(label2id)
else:
    classes_sorted = sorted(train_df[LABEL_COL].unique().tolist())
    label2id = {int(k): int(k) for k in classes_sorted}
    id2label = {int(k): str(int(k)) for k in classes_sorted}

# if val_df[LABEL_COL].dtype == object:
#     label2id = {lbl: i for i, lbl in enumerate(sorted(val_df[LABEL_COL].unique()))}
#     id2label = {i: lbl for lbl, i in label2id.items()}
#     val_df[LABEL_COL] = val_df[LABEL_COL].map(label2id)
# else:
#     classes_sorted = sorted(val_df[LABEL_COL].unique().tolist())
#     label2id = {int(k): int(k) for k in classes_sorted}
#     id2label = {int(k): str(int(k)) for k in classes_sorted}

num_labels = len(label2id)

# Train/validation split (stratified)
# train_df, val_df = train_test_split(
#     df, test_size=TEST_SIZE, stratify=df[LABEL_COL], random_state=RANDOM_STATE
# )

# Compute class weights from the TRAIN split only
classes = np.array(sorted(train_df[LABEL_COL].unique()))
class_weights = compute_class_weight(
    class_weight="balanced", classes=classes, y=train_df[LABEL_COL].values
)
# Put into index order 0..num_labels-1
# (If your classes aren't exactly 0..K-1, map them)
cw_full = np.zeros(num_labels, dtype=np.float32)
for cls, w in zip(classes, class_weights):
    cw_full[int(cls)] = float(w)
class_weights_tensor = torch.tensor(cw_full, dtype=torch.float)

print("Class weights:", cw_full)

# Convert to Hugging Face Datasets
train_ds = Dataset.from_pandas(train_df[[TEXT_COL, LABEL_COL]], preserve_index=False)
val_ds   = Dataset.from_pandas(val_df[[TEXT_COL, LABEL_COL]], preserve_index=False)
raw_datasets = DatasetDict({"train": train_ds, "validation": val_ds})

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tokenize(batch):
    return tokenizer(
        batch[TEXT_COL],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
    )

tokenized = raw_datasets.map(tokenize, batched=True)
tokenized = tokenized.rename_column(LABEL_COL, "labels")
tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# Model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

# Custom Trainer to apply class-weighted loss
class WeightedCELossTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # move to correct device once the model is placed
        self.register_buffer = None  # silence linters
        self.class_weights = class_weights

    # def compute_loss(self, model, inputs, return_outputs=False):
    #     labels = inputs.get("labels")
    #     outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
    #     logits = outputs.get("logits")

    #     # CrossEntropy with weights
    #     loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
    #     loss = loss_fct(logits, labels)
    #     return (loss, outputs) if return_outputs else loss
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
        logits = outputs.get("logits")
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

# Metrics (macro-F1)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    macro_f1 = f1_score(labels, preds, average="macro")
    # Optional: print per-class F1 in the end using classification_report
    return {"macro_f1": macro_f1, "accuracy": (preds == labels).mean()}

training_args = TrainingArguments(
    output_dir="./outputs",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),  # mixed precision if GPU supports it
    logging_steps=50,
    report_to="none",
)

trainer = WeightedCELossTrainer(
    class_weights=class_weights_tensor,
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

# Final evaluation with a detailed report
preds = trainer.predict(tokenized["validation"])
y_true = preds.label_ids
y_pred = preds.predictions.argmax(axis=1)

# If you mapped labels, show human-readable names if available
target_names = [id2label[i] for i in range(num_labels)]
print(classification_report(y_true, y_pred, target_names=target_names, digits=4))


Class weights: [ 0.72093683  0.29669908  1.4005994   2.5398254   8.75789    48.52732   ]


Map:   0%|          | 0/35522 [00:00<?, ? examples/s]

Map:   0%|          | 0/2512 [00:00<?, ? examples/s]

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_36/1286822756.py:118: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedCELossTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,1.039900,0.905527,0.508830,0.669188
2,0.763800,0.979096,0.547484,0.664411
3,0.631400,1.131426,0.546209,0.689889
4,0.478500,1.321409,0.536646,0.676354
5,0.392300,1.446515,0.530100,0.691481


                precision    recall  f1-score   support

       Abusive     0.5062    0.5780    0.5397       564
          None     0.8876    0.6637    0.7595      1451
Political Hate     0.4258    0.7491    0.5430       291
       Profane     0.6869    0.8662    0.7662       157
Religious Hate     0.3692    0.6316    0.4660        38
        Sexism     0.2500    0.1818    0.2105        11

      accuracy                         0.6644      2512
     macro avg     0.5209    0.6117    0.5475      2512
  weighted avg     0.7253    0.6644    0.6786      2512

